# Pràctica progressiva — Anàlisi de vendes d'una botiga esportiva `01_botiga_esport.csv`

**Nivell:** mitjà · **Durada orientativa:** 2–3 sessions de 90 minuts · **Format:** Jupyter Notebook

## Encàrrec professional
Treballes com a tècnic/a d'anàlisi de dades d'una botiga d'esport. Tens un CSV amb línies de venda i cal preparar les dades, justificar les regles aplicades i comunicar els resultats.

## Canvis respecte a la versió anterior
- El fitxer és `01_botiga_esport.csv`, amb els camps `article`, `familia_esportiva`, `botiga`, `unitats`, `distancia_km`, `data`.
- **No hi ha preu**, per tant no es pot calcular cap import ni facturació. La mètrica principal passa a ser **unitats venudes**.
- La capçalera i els valors tenen espais de farciment i el fitxer té BOM, de manera que cal llegir-lo amb `utf-8-sig` i netejar els noms de columna.

Utilitza només la biblioteca estàndard de Python.

## Fase 1 — Inspecció guiada

Comença per entendre el fitxer abans de modificar-lo. Cada fila és una línia de venda; els camps són textuals quan es llegeixen del CSV.

In [27]:
import csv
from datetime import datetime

nom_fitxer = "01_botiga_esport.csv"

with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    camps = [camp.strip() for camp in lector.fieldnames]
    registres = [
        {camp.strip() if camp is not None else None: valor for camp, valor in reg.items()}
        for reg in lector
    ]

print("Camps del CSV:", camps)
print("Nombre total de registres:", len(registres))
print("Exemple de primer registre:", registres[0])


Camps del CSV: ['article', 'familia_esportiva', 'botiga', 'unitats', 'distancia_km', 'data']
Nombre total de registres: 120
Exemple de primer registre: {'article': 'Pantalons curts', 'familia_esportiva': 'Material de raqueta', 'botiga': '', 'unitats': '1O', 'distancia_km': '97.6', 'data': '2025-08-27'}


### Exercici 1.1
Calcula quants registres tenen almenys un camp buit o format només per espais. No modifiquis les dades encara.

In [28]:
registres_incomplets = 0

for reg in registres:
    te_buit = any(val is None or str(val).strip() == "" for val in reg.values())
    if te_buit:
        registres_incomplets += 1

print(f"Registres amb almenys un camp buit/espais: {registres_incomplets}")

Registres amb almenys un camp buit/espais: 15


### Exercici 1.2

**Reflexiona:** quines dades són imprescindibles per calcular les unitats totals d'una venda? Quines són necessàries per segmentar-la (família, botiga)? Què passa amb `distancia_km`: és imprescindible?

## Fase 2 — Regles de qualitat i neteja guiada

Regles aplicades:
1. Elimina espais sobrants dels camps textuals (`strip()`).
2. `unitats` ha de ser un enter estrictament positiu (> 0). 
3. `distancia_km` ha de ser un nombre positiu (> 0); s'accepten coma o punt decimal. Valors com `gratis` es rebutgen.
4. La data ha de tenir el format `AAAA-MM-DD` i ser una data vàlida (es rebutgen `2025-02-29`, `2025/13/01`, `2025.06.10`...).
5. `article`, `familia_esportiva` i `botiga` són camps descriptius obligatoris.
6. Un registre amb valors numèrics o data no interpretables no entra als càlculs.

### Exercici 2 — Funcions de conversió amb gestió d'errors

In [29]:
def converteix_enter_positiu(valor: str):
    """Retorna un int positiu (>0) o None si no és vàlid."""
    try:
        v = int(str(valor).strip())
        return v if v > 0 else None
    except (ValueError, TypeError):
        return None

def converteix_decimal_positiu(valor: str):
    """Accepta coma o punt decimal; retorna float positiu (>0) o None si no és vàlid."""
    try:
        v = float(str(valor).strip().replace(",", "."))
        return v if v > 0 else None
    except (ValueError, TypeError):
        return None

def data_valida(valor: str) -> bool:
    """Retorna True si el text correspon a una data vàlida en format AAAA-MM-DD."""
    try:
        datetime.strptime(str(valor).strip(), "%Y-%m-%d")
        return True
    except (ValueError, TypeError):
        return False

### Exercici 3 — Neteja amb traçabilitat

- `vendes_netes`: diccionaris amb dades convertides.
- `rebutjades`: registres originals no vàlids.
- `incidencies`: detall per a la traçabilitat (`fila_csv`, `tipus`, `camp`, `valor_original`, `accio`).

In [30]:
vendes_netes = []
incidencies = []
rebutjades = []

obligatoris = ["article", "familia_esportiva", "botiga"]

for idx, reg in enumerate(registres, start=2):
    es_valid = True
    reg_net = {}

    # 1. Camps descriptius obligatoris
    for camp in obligatoris:
        val_original = reg.get(camp,"")
        val_net = val_original.strip()
        if not val_net:
            es_valid = False
            incidencies.append({"fila_csv": idx, "tipus": "REBUTJAT", "camp": camp,
                                "valor_original": val_original, "accio": "Camp obligatori buit"})
        else:
            if val_original != val_net:
                incidencies.append({"fila_csv": idx, "tipus": "CORRECCIO", "camp": camp,
                                    "valor_original": val_original, "accio": "Espais sobrants eliminats"})
            reg_net[camp] = val_net

    # 2. Conversió d'unitats
    unitats_original = reg.get("unitats")
    unitats_val = converteix_enter_positiu(unitats_original)
    if unitats_val is None:
        es_valid = False
        incidencies.append({"fila_csv": idx, "tipus": "REBUTJAT", "camp": "unitats",
                            "valor_original": unitats_original, "accio": "Enter no positiu o no vàlid"})
    else:
        reg_net["unitats"] = unitats_val

    # 3. Distància (km)
    dist_original = reg.get("distancia_km")
    dist_val = converteix_decimal_positiu(dist_original)
    if dist_val is None:
        es_valid = False
        incidencies.append({"fila_csv": idx, "tipus": "REBUTJAT", "camp": "distancia_km",
                            "valor_original": dist_original, "accio": "Distància no positiva o no numèrica"})
    else:
        if "," in dist_original:
            incidencies.append({"fila_csv": idx, "tipus": "CORRECCIO", "camp": "distancia_km",
                                "valor_original": dist_original, "accio": "Coma convertida a punt decimal"})
        reg_net["distancia_km"] = dist_val

    # 4. Validació de data
    data_orig = str(reg.get("data", "")).strip()
    if not data_valida(data_orig):
        es_valid = False
        incidencies.append({"fila_csv": idx, "tipus": "REBUTJAT", "camp": "data",
                            "valor_original": data_orig, "accio": "Data amb format incorrecte o no vàlida"})
    else:
        reg_net["data"] = data_orig

 # Classificació final del registre
    if es_valid:
        vendes_netes.append(reg_net)
    else:
        rebutjades.append(reg)

print(f"Llegits: {len(registres)}")
print(f"Vàlids: {len(vendes_netes)}")
print(f"Rebutjats: {len(rebutjades)}")
print(f"Incidències documentades: {len(incidencies)}")

Llegits: 120
Vàlids: 90
Rebutjats: 30
Incidències documentades: 57


## Fase 3 — Indicadors de negoci

Sense preu no hi ha facturació: els indicadors es basen en **unitats** i **distància**.

In [31]:
total_unitats = sum(v["unitats"] for v in vendes_netes)
unitats_mitjanes = round(total_unitats / len(vendes_netes), 2) if vendes_netes else 0.0
distancia_mitjana = round(sum(v["distancia_km"] for v in vendes_netes) / len(vendes_netes), 2) if vendes_netes else 0.0

resum_familia = {}
resum_botiga = {}

for v in vendes_netes:
    resum_familia[v["familia_esportiva"]] = resum_familia.get(v["familia_esportiva"], 0) + v["unitats"]
    resum_botiga[v["botiga"]] = resum_botiga.get(v["botiga"], 0) + v["unitats"]

print(f"Total unitats: {total_unitats}")
print(f"Unitats mitjanes per línia: {unitats_mitjanes}")
print(f"Distància mitjana: {distancia_mitjana} km")
print("Unitats per família:", resum_familia)
print("Unitats per botiga:", resum_botiga)

Total unitats: 3669
Unitats mitjanes per línia: 40.77
Distància mitjana: 127.25 km
Unitats per família: {'Accessoris': 446, 'Material de raqueta': 766, 'Roba': 666, 'Pilotes': 892, 'Calçat': 899}
Unitats per botiga: {'Lleida': 591, 'Manresa': 908, 'Barcelona': 537, 'Girona': 949, 'Tarragona': 684}


### Control de qualitat

In [32]:
assert len(vendes_netes) + len(rebutjades) == len(registres), "La suma de vàlids i rebutjats no quadra amb el total."

assert all(v["unitats"] > 0 and v["distancia_km"] > 0 for v in vendes_netes), "Hi ha valors nuls o negatius."

assert sum(resum_familia.values()) == total_unitats, "La suma per família no coincideix amb el total d'unitats."
assert sum(resum_botiga.values()) == total_unitats, "La suma per botiga no coincideix amb el total d'unitats."

print("Tots els controls de qualitat s'han superat amb èxit!")

Tots els controls de qualitat s'han superat amb èxit!


## Fase 4 — Interpretació i comunicació

In [33]:
top_familia = max(resum_familia.items(), key=lambda x: x[1]) if resum_familia else (None, 0)
top_botiga = max(resum_botiga.items(), key=lambda x: x[1]) if resum_botiga else (None, 0)

unitats_per_article = {}
for v in vendes_netes:
    unitats_per_article[v["article"]] = unitats_per_article.get(v["article"], 0) + v["unitats"]
top_article = max(unitats_per_article.items(), key=lambda x: x[1]) if unitats_per_article else (None, 0)

print(f"1. Família amb més unitats: {top_familia[0]} ({top_familia[1]} unitats)")
print(f"   Botiga amb més unitats: {top_botiga[0]} ({top_botiga[1]} unitats)")
print(f"2. Article amb més unitats venudes: {top_article[0]} ({top_article[1]} unitats)")

1. Família amb més unitats: Calçat (899 unitats)
   Botiga amb més unitats: Girona (949 unitats)
2. Article amb més unitats venudes: Raqueta (712 unitats)


## Fase 5 — Lliurament professional

Generació dels tres fitxers de sortida: `informe_vendes.txt`, `incidencies.csv` i `vendes_netes.csv`.

In [34]:
nom_informe = "informe_vendes.txt"
nom_incidencies = "incidencies.csv"
nom_netes = "vendes_netes.csv"

# 1. vendes_netes.csv
if vendes_netes:
    with open(nom_netes, mode="w", encoding="utf-8", newline="") as f:
        escriptor = csv.DictWriter(f, fieldnames=list(vendes_netes[0].keys()))
        escriptor.writeheader()
        escriptor.writerows(vendes_netes)

# 2. incidencies.csv
with open(nom_incidencies, mode="w", encoding="utf-8", newline="") as f:
    camps_inc = ["fila_csv", "tipus", "camp", "valor_original", "accio"]
    escriptor = csv.DictWriter(f, fieldnames=camps_inc)
    escriptor.writeheader()
    escriptor.writerows(incidencies)

# 3. informe_vendes.txt
text_informe = f"""=========================================
INFORME EXECUTIU DE VENDES
=========================================
Data de generació: {datetime.now().strftime('%Y-%m-%d %H:%M')}
Cobertura: {len(vendes_netes)}/{len(registres)} registres vàlids ({len(rebutjades)} rebutjats)

INDICADORS GLOBALS:
- Total unitats venudes: {total_unitats}
- Unitats mitjanes per línia: {unitats_mitjanes:.2f}
- Distància mitjana: {distancia_mitjana:.2f} km

UNITATS PER FAMÍLIA:
"""
for fam, u in resum_familia.items():
    text_informe += f"- {fam}: {u} unitats\n"

text_informe += "\nUNITATS PER BOTIGA:\n"
for bot, u in resum_botiga.items():
    text_informe += f"- {bot}: {u} unitats\n"

with open(nom_informe, mode="w", encoding="utf-8") as f:
    f.write(text_informe)

print("Fitxers d'informe i exportació generats correctament.")

Fitxers d'informe i exportació generats correctament.
